# UFO Analysis & Visualization

In [78]:
import pandas as pd
import altair as alt
import reverse_geocoder as rg
from pycountry_convert import country_alpha2_to_continent_code
import pycountry

In [55]:
ufo = pd.read_csv('ufo-scrubbed-geocoded-time-standardized-00.csv')
ufo.head()

,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 1949-50. It occurred after a Boy Scout meeting in the Baptist Church. The Baptist Church sit,4/27/2004,29.8830556,-97.9411111
0,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
1,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
2,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
3,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611
4,10/10/1961 19:00,bristol,tn,us,sphere,300.0,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889


In [56]:
# copied column names from HW3, with some edits
cols = ['date', 'city', 'state', 'country', 'shape', 'duration_seconds',
       'duration_text', 'comment', 'report_date', 'latitude', 'longitude']

ufo = pd.read_csv('ufo-scrubbed-geocoded-time-standardized-00.csv', header=None, names=cols)
ufo.head()

,date,city,state,country,shape,duration_seconds,duration_text,comment,report_date,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


## Cleaning the dataset

1. Turning date and time strings into datetime values
2. Deal with missing values differently
3.

In [57]:
ufo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              80332 non-null  object 
 1   city              80332 non-null  object 
 2   state             74535 non-null  object 
 3   country           70662 non-null  object 
 4   shape             78400 non-null  object 
 5   duration_seconds  80332 non-null  float64
 6   duration_text     80332 non-null  object 
 7   comment           80317 non-null  object 
 8   report_date       80332 non-null  object 
 9   latitude          80332 non-null  float64
 10  longitude         80332 non-null  float64
dtypes: float64(3), object(8)
memory usage: 6.7+ MB


In [58]:
# parsing dates
ufo['date'] = pd.to_datetime(ufo['date'], format='mixed')
ufo['report_date'] = pd.to_datetime(ufo['report_date'], format='mixed')

In [59]:
# extracting useful time columns for plotting
ufo['year'] = ufo['date'].dt.year
ufo['month'] = ufo['date'].dt.month
ufo['day'] = ufo['date'].dt.day
ufo['hour'] = ufo['date'].dt.hour
ufo['day_of_week'] = ufo['date'].dt.day_name()
ufo['month_name'] = ufo['date'].dt.month_name()

Geography

In [60]:
#: :generated:
#: :model: Claude Opus 4.6
#: :prompt: Use reverse_geocoder to fill in country and continent for every row based on latitude and longitude. Add country code, full country name, and continent columns.
#: :changes: None
#: :response:

# import reverse_geocoder as rg
# import pycountry
# from pycountry_convert import country_alpha2_to_continent_code

# Reverse geocode all coordinates
coords = list(zip(ufo['latitude'], ufo['longitude']))
results = rg.search(coords)

# Extract country code
ufo['country_rg'] = [r['cc'] for r in results]

# Map country code to full country name
def get_country_name(cc):
    try:
        return pycountry.countries.get(alpha_2=cc).name
    except:
        return 'Unknown'

ufo['country_name'] = ufo['country_rg'].apply(get_country_name)

# Map country code to continent
continent_map = {
    'NA': 'North America',
    'SA': 'South America',
    'EU': 'Europe',
    'AF': 'Africa',
    'AS': 'Asia',
    'OC': 'Oceania',
    'AN': 'Antarctica'
}

def get_continent(cc):
    try:
        return continent_map[country_alpha2_to_continent_code(cc)]
    except:
        return 'Unknown'

ufo['continent'] = ufo['country_rg'].apply(get_continent)
#: :end:

In [61]:
ufo.head(20)

,date,city,state,country,shape,duration_seconds,duration_text,comment,report_date,latitude,longitude,year,month,day,hour,day_of_week,month_name,country_rg,country_name,continent
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,2004-04-27,29.883056,-97.941111,1949,10,10,20,Monday,October,US,United States,North America
1,1949-10-10 21:00:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,2005-12-16,29.384210,-98.581082,1949,10,10,21,Monday,October,US,United States,North America
2,1955-10-10 17:00:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,2008-01-21,53.200000,-2.916667,1955,10,10,17,Monday,October,GB,United Kingdom,Europe
3,1956-10-10 21:00:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,2004-01-17,28.978333,-96.645833,1956,10,10,21,Wednesday,October,US,United States,North America
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,2004-01-22,21.418056,-157.803611,1960,10,10,20,Monday,October,US,United States,North America
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300.0,5 minutes,My father is now 89 my brother 52 the girl wit...,2007-04-27,36.595000,-82.188889,1961,10,10,19,Tuesday,October,US,United States,North America
6,1965-10-10 21:00:00,penarth (uk/wales),NaN,gb,circle,180.0,about 3 mins,penarth uk circle 3mins stayed 30ft above m...,2006-02-14,51.434722,-3.180000,1965,10,10,21,Sunday,October,GB,United Kingdom,Europe
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200.0,20 minutes,A bright orange color changing to reddish colo...,1999-10-02,41.117500,-73.408333,1965,10,10,23,Sunday,October,US,United States,North America
8,1966-10-10 20:00:00,pell city,al,us,disk,180.0,3 minutes,Strobe Lighted disk shape object observed clos...,2009-03-19,33.586111,-86.286111,1966,10,10,20,Monday,October,US,United States,North America
9,1966-10-10 21:00:00,live oak,fl,us,disk,120.0,several minutes,Saucer zaps energy from powerline as my pregna...,2005-05-11,30.294722,-82.984167,1966,10,10,21,Monday,October,US,United States,North America


Still missing a few rows..

States, comments, and shapes. I don't really care about comments, so I'll only be dealing with states and shapes only.

In [62]:
# check which countries have states
has_state = ufo[ufo["state"].notna()].groupby("country_rg").agg(
    rows_with_state=("state", "count"),
    sample_states=("state", lambda x: ", ".join(x.unique()[:5]))
).sort_values("rows_with_state", ascending=False)

has_state.head(20)

,rows_with_state,sample_states
country_rg,,
US,70816,"tx, hi, tn, ct, al"
CA,3567,"ab, on, nf, bc, mb"
PR,32,pr
MX,20,"ca, tx, al, de, mb"
GB,17,"wv, la, de, bc, tn"
AU,13,"nt, wa, dc, al, yt"
BR,9,"al, pa, pe, ak, fl"
IN,9,"ny, or, ct, az, in"
ZA,7,"yt, ar, al, nc"


The US has the most states in this dataset. 70K will make for a nice plot, so I'll be dealing with states in the US only.

In [63]:
#: :generated:
#: :model: Claude Opus 4.6
#: :prompt: Capitalize US state abbreviations and add a column with full state names. Only map state names for US rows.
#: :changes: None
#: :response:
ufo["state"] = ufo["state"].str.upper()

us_states = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming", "DC": "District of Columbia"
}

ufo["state_name"] = ufo["state"].where(ufo["country_rg"] == "US").map(us_states)
#: :end:

In [64]:
us_rows = ufo[ufo["country_rg"] == "US"]
print(f"US rows total: {len(us_rows)}")
print(f"US rows with state: {us_rows['state'].notna().sum()}")
print(f"US rows missing state: {us_rows['state'].isna().sum()}")

US rows total: 70904
US rows with state: 70816
US rows missing state: 88


Filling missing US states with the long/lat valued using reverse_geocoder

In [65]:
# fill missing US states using reverse geocoder results
missing_mask = (ufo["country_rg"] == "US") & (ufo["state"].isna())

missing_coords = list(zip(
    ufo.loc[missing_mask, "latitude"],
    ufo.loc[missing_mask, "longitude"]
))
results = rg.search(missing_coords)

# state abbreviation
state_to_abbr = {v: k for k, v in us_states.items()}
ufo.loc[missing_mask, "state"] = [state_to_abbr.get(r["admin1"], None) for r in results]

In [66]:
us_rows = ufo[ufo["country_rg"] == "US"]
print(f"US rows total: {len(us_rows)}")
print(f"US rows with state: {us_rows['state'].notna().sum()}")
print(f"US rows missing state: {us_rows['state'].isna().sum()}")

US rows total: 70904
US rows with state: 70904
US rows missing state: 0


Now shapes..

In [73]:
print(ufo['shape'].unique())

['cylinder' 'light' 'circle' 'sphere' 'disk' 'fireball' 'unknown' 'oval'
 'other' 'cigar' 'rectangle' 'chevron' 'triangle' 'formation' nan 'delta'
 'changing' 'egg' 'diamond' 'flash' 'teardrop' 'cone' 'cross' 'pyramid'
 'round' 'crescent' 'flare' 'hexagon' 'dome' 'changed']


In [74]:
ufo['shape'] = ufo['shape'].fillna('unknown')

In [75]:
ufo.head()

,date,city,state,country,shape,duration_seconds,duration_text,comment,report_date,latitude,...,year,month,day,hour,day_of_week,month_name,country_rg,country_name,continent,state_name
0,1949-10-10 20:30:00,san marcos,TX,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,2004-04-27,29.883056,...,1949,10,10,20,Monday,October,US,United States,North America,Texas
1,1949-10-10 21:00:00,lackland afb,TX,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,2005-12-16,29.384210,...,1949,10,10,21,Monday,October,US,United States,North America,Texas
2,1955-10-10 17:00:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,2008-01-21,53.200000,...,1955,10,10,17,Monday,October,GB,United Kingdom,Europe,NaN
3,1956-10-10 21:00:00,edna,TX,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,2004-01-17,28.978333,...,1956,10,10,21,Wednesday,October,US,United States,North America,Texas
4,1960-10-10 20:00:00,kaneohe,HI,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,2004-01-22,21.418056,...,1960,10,10,20,Monday,October,US,United States,North America,Hawaii


## First Plot, static

I wanted to create a similar plot to the [Sighting Time, Seasonally from this inforgraph](https://www.flerlagetwins.com/2017/02/ufo-sightings-of-2016-recreation-of_99.html).

In [82]:
# some preparations
month_hour = ufo.groupby(["month","month_name","hour"]).size().reset_index(name="count")
month_totals = month_hour.groupby("month")["count"].transform("sum")
month_hour["pct"] = (month_hour["count"] / month_totals * 100).round(1)
month_hour["pct_label"] = month_hour["pct"].astype(str) + "%"

hour_totals = ufo.groupby("hour").size().reset_index(name="count")
month_order = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]

In [ ]:
# bar chart
bars = alt.Chart(hour_totals).mark_bar().encode(
    x=alt.X("hour:O", title=None, axis=alt.Axis(labels=False, ticks=False, domain=False)),
    y=alt.Y("count:Q", axis=None),
    tooltip=[alt.Tooltip("hour:O", title="Hour"), alt.Tooltip("count:Q", title="Sightings", format=",")]
).properties(width=650, height=80)

In [ ]:
# heatmap
heatmap = alt.Chart(month_hour).mark_rect().encode(
    x=alt.X("hour:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("month_name:N", sort=month_order, title=None),
    color=alt.Color("pct:Q", scale=alt.Scale(scheme="blues"), legend=None),
    tooltip=[
        alt.Tooltip("month_name:N", title="Month"),
        alt.Tooltip("hour:O", title="Hour"),
        alt.Tooltip("pct_label:N", title="% of Month"),
        alt.Tooltip("count:Q", title="Sightings", format=",")
    ]
).properties(width=650, height=320)

In [ ]:
# text labels
text = alt.Chart(month_hour).mark_text(fontSize=9).encode(
    x=alt.X("hour:O"),
    y=alt.Y("month_name:N", sort=month_order),
    text="pct_label:N",
    # making sure all text is readable
    color=alt.condition(alt.datum.pct > 12, alt.value("white"), alt.value("#333333"))
)

In [99]:
heatmap_ufo = alt.vconcat(
    bars, heatmap + text, spacing=5
).configure_view(strokeWidth=0
).properties(title=alt.Title("Sighting Time, Seasonally", fontSize=18, anchor="middle"))

In [106]:
heatmap_ufo

alt.VConcatChart(...)

In [109]:
heatmap_ufo.save("heatmap.json")

## Second Plot

In [90]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [ ]:
# aggregate
ufo["lat_round"] = ufo["latitude"].round(1)
ufo["lon_round"] = ufo["longitude"].round(1)
ufo["decade"] = (ufo["year"] // 10 * 10).astype(str) + "s"

agg = ufo.groupby(["lat_round", "lon_round", "decade", "continent"]).size().reset_index(name="count")
decade_counts = ufo.groupby("decade").size().reset_index(name="total")

In [ ]:
# selections
continent_selection = alt.selection_point(fields=["continent"], bind="legend")
decade_selection = alt.selection_interval(encodings=["x"])

In [ ]:
# background
sphere = alt.Chart(alt.sphere()).mark_geoshape(fill="#f0f0f0", stroke="white")
graticule = alt.Chart(alt.graticule()).mark_geoshape(stroke="#ddd", strokeWidth=0.5, filled=False)
countries = alt.Chart(
    alt.topo_feature("https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json", "countries")
).mark_geoshape(fill="#e8e8e8", stroke="white", strokeWidth=0.5)

In [ ]:
# points
points = alt.Chart(agg).mark_circle().encode(
    longitude="lon_round:Q",
    latitude="lat_round:Q",
    size=alt.Size("count:Q", scale=alt.Scale(range=[2, 80]), legend=None),
    color=alt.Color("continent:N", title="Continent"),
    opacity=alt.condition(continent_selection, alt.value(0.4), alt.value(0.02)),
    tooltip=[
        alt.Tooltip("lat_round:Q", title="Latitude"),
        alt.Tooltip("lon_round:Q", title="Longitude"),
        alt.Tooltip("continent:N", title="Continent"),
        alt.Tooltip("count:Q", title="Sightings", format=",")
    ]
).transform_filter(
    decade_selection
).transform_aggregate(
    count="sum(count)", groupby=["lat_round", "lon_round", "continent"]
).add_params(continent_selection)

map_chart = (sphere + graticule + countries + points).project(
    "equalEarth"
).properties(width=800, height=450)


In [ ]:
# decade selector strip
decade_strip = alt.Chart(decade_counts).mark_bar(
    cornerRadiusTopLeft=3, cornerRadiusTopRight=3, cursor="pointer"
).encode(
    x=alt.X("decade:N", sort=sorted(decade_counts["decade"].unique().tolist()),
            title=None, axis=alt.Axis(labelAngle=0, labelFontSize=12)),
    y=alt.Y("total:Q", title=None, axis=None, scale=alt.Scale(type="sqrt")),
    color=alt.condition(decade_selection, alt.value("#4c78a8"), alt.value("#ddd")),
    tooltip=[
        alt.Tooltip("decade:N", title="Decade"),
        alt.Tooltip("total:Q", title="Total Sightings", format=",")
    ]
).add_params(decade_selection
).properties(width=800, height=60, title="Click a decade to filter")

In [103]:
map_ufo = alt.vconcat(
    map_chart, decade_strip, spacing=10
).properties(
    title=alt.Title("UFO Sightings Around the World", fontSize=18, anchor="middle")
).configure_view(strokeWidth=0)

In [105]:
map_ufo

alt.VConcatChart(...)

In [108]:
map_ufo.save("map.json")